# Shared enums - JavaScript

All 9 JavaScript examples from [docs/enums.md](https://platob.github.io/yggdryl/enums/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { DataType } = require('yggdryl')

const value = new DataType('int64')
assert.equal(value.id, 'int64')
assert.equal(value.kind, 'integer')

## Identity carries no parameters

In [ ]:
const assert = require('node:assert/strict')
const { DataType } = require('yggdryl')

const stamp = new DataType('timestamp(us, UTC)')
assert.equal(stamp.id, 'timestamp')
assert.equal(stamp.toString(), 'timestamp(us,"UTC")')

## MIME types

In [ ]:
const assert = require('node:assert/strict')
const { MimeType } = require('yggdryl')

const parquet = MimeType.fromExtension('parquet')
assert.ok(parquet.equals(MimeType.PARQUET))
assert.equal(parquet.toString(), 'application/vnd.apache.parquet')
assert.equal(parquet.topLevel, 'application')
assert.ok(parquet.isTabular() && parquet.isBinary())

const custom = new MimeType('Application/Vnd.Example+JSON')
assert.equal(custom.toString(), 'application/vnd.example+json')
assert.equal(custom.structuredSuffix, 'json')
assert.equal(custom.isKnown(), false)
assert.equal(custom.isStructured(), true)

In [ ]:
const assert = require('node:assert/strict')
const { MimeType } = require('yggdryl')

assert.ok(
  MimeType.fromContentType('Application/JSON; charset="utf-8"').equals(MimeType.JSON),
)
assert.throws(() => MimeType.fromContentType('application/json; charset'))

assert.ok(MimeType.fromContentCoding('x-gzip').equals(MimeType.GZIP))
assert.equal(MimeType.GZIP.contentCoding, 'gzip')
assert.throws(() => MimeType.fromContentCoding('identity'))

## Media types are a base plus its codings

In [ ]:
const assert = require('node:assert/strict')
const { MediaType, MimeType } = require('yggdryl')

const media = MediaType.fromFileName('trades.json.gz')
assert.ok(media.base.equals(MimeType.JSON))
assert.deepEqual(media.encodings.map((value) => value.toString()), ['application/gzip'])
assert.ok(media.encoding.equals(MimeType.GZIP))
assert.deepEqual(media.extensions, ['json', 'gz'])
assert.equal(media.isEncoded(), true)
assert.equal(media.toString(), 'application/json;encodings=application/gzip')

In [ ]:
const assert = require('node:assert/strict')
const { MediaType, MimeType } = require('yggdryl')

const media = MediaType.fromContentHeaders('text/csv; charset=utf-8', 'gzip')
assert.ok(media.base.equals(MimeType.CSV))
assert.ok(media.encoding.equals(MimeType.GZIP))

assert.ok(
  MediaType.fromExtension('tgz').equals(
    MediaType.fromParts(MimeType.TAR, [MimeType.GZIP]),
  ),
)

const stacked = MediaType.fromFileName('events.json')
assert.throws(() => stacked.pushEncoding(MimeType.ZIP))
stacked.pushEncoding(MimeType.ZSTD)
assert.deepEqual(stacked.extensions, ['json', 'zst'])

## Time units and union modes

In [ ]:
const assert = require('node:assert/strict')
const { DataType } = require('yggdryl')

assert.equal(DataType.time('microseconds').toString(), 'time64(us)')
assert.equal(DataType.time('MICRO SECONDS').toString(), 'time64(us)')
assert.equal(DataType.time('s').toString(), 'time32(s)')

## Listing the vocabularies

In [ ]:
const assert = require('node:assert/strict')
const { enums } = require('yggdryl')

assert.ok(enums.dataTypeIds.includes('int64'))
assert.deepEqual([...enums.unionModes], ['sparse', 'dense'])
assert.ok(enums.timeUnits.includes('us'))
assert.ok(enums.codecs.includes('gzip'))
assert.ok(enums.ioKinds.includes('file'))
assert.ok(enums.compatibilitySchemes.includes('arrow'))
assert.equal(enums.levels.default, 6)

## Timezone

In [ ]:
const assert = require('node:assert/strict')
const { Timezone } = require('yggdryl')

// A name, an alias, and a zone read out of Intl all arrive at one value.
assert.ok(Timezone.from('Asia/Calcutta').equals(Timezone.from('Asia/Kolkata')))
assert.equal(Timezone.from('US/Eastern').key, 'America/New_York')
assert.ok(Timezone.from('Z').equals(Timezone.UTC))

const sydney = Timezone.from('Australia/Sydney')
// Sydney's saving period spans the new year, so January is +11.
assert.equal(sydney.offsetAt(1_705_000_000), 11 * 3600)
assert.equal(sydney.offsetAt(1_720_000_000), 10 * 3600)
assert.ok(sydney.observesSaving())

// It reports an offset the way `Date` does wherever that is what is read.
assert.equal(sydney.getTimezoneOffset(1_720_000_000), -600)